In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Stage 0: tokenize a TinyStories subset into uint16 .bin files
# Kaggle: Settings -> Internet ON, Accelerator = None (CPU is enough). ~5-10 min.

!pip install -q tiktoken

import numpy as np
import tiktoken
from datasets import load_dataset

TRAIN_TOKENS = 30_000_000
VAL_TOKENS = 1_000_000
OUT = "/kaggle/working"

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token  # 50256 -> fits in uint16 (max 65535)


def build(split, target, path):
    # streaming: we only download what we need, not the full dataset
    ds = load_dataset("roneneldan/TinyStories", split=split, streaming=True)
    buf = np.empty(target, dtype=np.uint16)
    n = 0
    stories = 0
    batch = []

    for ex in ds:
        batch.append(ex["text"])
        if len(batch) < 1000:
            continue
        for ids in enc.encode_ordinary_batch(batch):
            ids = ids + [EOT]  # EOT separates stories
            k = min(len(ids), target - n)
            buf[n:n + k] = ids[:k]
            n += k
            stories += 1
            if n == target:
                break
        batch = []
        if n == target:
            break

    buf[:n].tofile(path)
    print(f"{split}: {n:,} tokens from {stories:,} stories "
          f"(~{n / stories:.0f} tokens/story) -> {path}")
    return buf[:n]


train = build("train", TRAIN_TOKENS, f"{OUT}/train.bin")
val = build("validation", VAL_TOKENS, f"{OUT}/val.bin")

# sanity checks
assert train.max() < enc.n_vocab and val.max() < enc.n_vocab
print("vocab size:", enc.n_vocab)
print("\n--- first 150 train tokens decoded ---")
print(enc.decode(train[:150].tolist()))

README.md: 0.00B [00:00, ?B/s]

train: 30,000,000 tokens from 135,157 stories (~222 tokens/story) -> /kaggle/working/train.bin
validation: 1,000,000 tokens from 4,999 stories (~200 tokens/story) -> /kaggle/working/val.bin
vocab size: 50257

--- first 150 train tokens decoded ---
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt.
